In [2]:
!pip install pyhealth

In [4]:
from pyhealth.datasets import MIMIC3Dataset
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!ls "/content/drive/MyDrive"

'Colab Notebooks'  'mimic-iii-clinical-database-1.4 3'


In [8]:
!ls "/content/drive/MyDrive/mimic-iii-clinical-database-1.4 3"

ADMISSIONS.csv.gz	   D_ICD_PROCEDURES.csv.gz    OUTPUTEVENTS.csv.gz
CALLOUT.csv.gz		   D_ITEMS.csv.gz	      PATIENTS.csv.gz
CAREGIVERS.csv.gz	   D_LABITEMS.csv.gz	      PRESCRIPTIONS.csv.gz
CHARTEVENTS.csv.gz	   DRGCODES.csv.gz	      PROCEDUREEVENTS_MV.csv.gz
checksum_md5_unzipped.txt  ICUSTAYS.csv.gz	      PROCEDURES_ICD.csv.gz
checksum_md5_zipped.txt    INPUTEVENTS_CV.csv.gz      README.md
CPTEVENTS.csv.gz	   INPUTEVENTS_MV.csv.gz      SERVICES.csv.gz
DATETIMEEVENTS.csv.gz	   LABEVENTS.csv.gz	      SHA256SUMS.txt
D_CPT.csv.gz		   LICENSE.txt		      TRANSFERS.csv.gz
DIAGNOSES_ICD.csv.gz	   MICROBIOLOGYEVENTS.csv.gz
D_ICD_DIAGNOSES.csv.gz	   NOTEEVENTS.csv.gz


In [10]:
mimic_dataset = MIMIC3Dataset(
    root="/content/drive/MyDrive/mimic-iii-clinical-database-1.4 3",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    dev=False
)

mimic_dataset.stats()

No config path provided, using default config


INFO:pyhealth.datasets.mimic3:No config path provided, using default config


Initializing mimic3 dataset from /content/drive/MyDrive/mimic-iii-clinical-database-1.4 3 (dev mode: False)


INFO:pyhealth.datasets.base_dataset:Initializing mimic3 dataset from /content/drive/MyDrive/mimic-iii-clinical-database-1.4 3 (dev mode: False)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39


...

Joining with table: /content/drive/MyDrive/mimic-iii-clinical-database-1.4 3/ADMISSIONS.csv.gz


INFO:pyhealth.datasets.base_dataset:Joining with table: /content/drive/MyDrive/mimic-iii-clinical-database-1.4 3/ADMISSIONS.csv.gz


Caching event dataframe to /root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/global_event_df.parquet...


INFO:pyhealth.datasets.base_dataset:Caching event dataframe to /root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/global_event_df.parquet...


Dataset: mimic3
Dev mode: False
Number of patients: 46520
Number of events: 5214620


In [12]:
from pyhealth.tasks import ReadmissionPredictionMIMIC3
import datetime as dt
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import RNN
from pyhealth.trainer import Trainer

In [16]:
W = dt.timedelta(days=30)

from pyhealth.utils import set_seed

set_seed(42)


readmission_task = ReadmissionPredictionMIMIC3(window=W, exclude_minors=False)
sample_data = mimic_dataset.set_task(task=readmission_task)

# STEP 3: Split and create dataloaders
train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_data, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)



Setting task ReadmissionPredictionMIMIC3 for mimic3 base dataset...


INFO:pyhealth.datasets.base_dataset:Setting task ReadmissionPredictionMIMIC3 for mimic3 base dataset...


Task cache paths: task_df=/root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/tasks/ReadmissionPredictionMIMIC3_960b0303-8340-5da2-8e7d-8d90c65aa6e5/task_df.ld, samples=/root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/tasks/ReadmissionPredictionMIMIC3_960b0303-8340-5da2-8e7d-8d90c65aa6e5/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Task cache paths: task_df=/root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/tasks/ReadmissionPredictionMIMIC3_960b0303-8340-5da2-8e7d-8d90c65aa6e5/task_df.ld, samples=/root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/tasks/ReadmissionPredictionMIMIC3_960b0303-8340-5da2-8e7d-8d90c65aa6e5/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Found cached processed samples at /root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/tasks/ReadmissionPredictionMIMIC3_960b0303-8340-5da2-8e7d-8d90c65aa6e5/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.


INFO:pyhealth.datasets.base_dataset:Found cached processed samples at /root/.cache/pyhealth/e332267b-17d7-5ec5-a75d-e67364b85a39/tasks/ReadmissionPredictionMIMIC3_960b0303-8340-5da2-8e7d-8d90c65aa6e5/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.


In [17]:
# STEP 4: Define model
model = RNN(
  dataset=sample_data,
)

# STEP 5: Train
trainer = Trainer(model=model)
trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=1,
    monitor="roc_auc",
)

# STEP 6: Evaluate
trainer.evaluate(test_dataloader)

RNN(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(4102, 128, padding_idx=0)
    (procedures): Embedding(1304, 128, padding_idx=0)
    (drugs): Embedding(2624, 128, padding_idx=0)
  ))
  (rnn): ModuleDict(
    (conditions): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (procedures): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (drugs): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
  )
  (fc): Linear(in_features=384, out_features=1, bias=True)
)


INFO:pyhealth.trainer:RNN(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(4102, 128, padding_idx=0)
    (procedures): Embedding(1304, 128, padding_idx=0)
    (drugs): Embedding(2624, 128, padding_idx=0)
  ))
  (rnn): ModuleDict(
    (conditions): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (procedures): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (drugs): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
  )
  (fc): Linear(in_features=384, out_features=1, bias=True)
)


Metrics: None


INFO:pyhealth.trainer:Metrics: None


Device: cpu


...

New best roc_auc score (0.6262) at epoch-0, step-239


INFO:pyhealth.trainer:New best roc_auc score (0.6262) at epoch-0, step-239


Loaded best model


INFO:pyhealth.trainer:Loaded best model
Evaluation: 100%|██████████| 31/31 [00:02<00:00, 13.55it/s]


{'pr_auc': 0.33600554262775284,
 'roc_auc': 0.575195641084085,
 'f1': 0.05090909090909091,
 'loss': 0.5808628412985033}